# Ćwiczenie 4.2: progi liczbowe i drzewo regresyjne

Ten notebook odpowiada na pytanie: co się zmienia, gdy cecha nie jest już binarna jak `cieplo`, tylko liczbowa jak `projekt_pkt` albo `godziny_nauki`?

Krótka odpowiedź: drzewo nadal zadaje pytania typu tak/nie, ale samo szuka progu:

```text
projekt_pkt <= 59.5?
godziny_nauki <= 6.5?
```

W drugiej części przechodzimy od drzewa klasyfikacyjnego do regresyjnego. Tam liść nie zwraca klasy, tylko liczbę, zwykle średnią z obserwacji w liściu.

Plik: **wersja dla studentów**.


## 0. Most z notebooka 4.1: od pytań gotowych do pytań szukanych

W poprzednim notebooku wszystkie cechy były binarne, np. `cieplo`, `weekend`, `krotka_kolejka`.

To oznaczało, że pytania były gotowe:

```text
cieplo = 1?          TAK / NIE
weekend = 1?         TAK / NIE
krotka_kolejka = 1?  TAK / NIE
```

W tym notebooku pojawiają się cechy liczbowe/punktowe, np. liczba punktów z projektu albo liczba godzin nauki.

Nie chcemy tworzyć osobnej gałęzi dla każdej wartości:

```text
projekt_pkt = 35?
projekt_pkt = 42?
projekt_pkt = 48?
...
```

Drzewo zamienia więc cechę liczbową na pytanie binarne przez dobranie progu:

```text
projekt_pkt <= 59.5?  TAK / NIE
```

Po znalezieniu progu mechanika jest taka sama jak wcześniej: liczymy jakość podziału i wybieramy najlepszy split.


## 1. Dane klasyfikacyjne: zaliczenie

Mamy mały zbiór studentów. Celem jest przewidzenie, czy student zaliczył.

Cechy są liczbowe:

- `projekt_pkt`,
- `quiz_pkt`,
- `zadania_pkt`,
- `obecnosc_pct`.

Drzewo nie pyta już `cecha == 0/1`, tylko testuje wiele progów, np. `projekt_pkt <= 59.5`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
from sklearn.metrics import accuracy_score, mean_squared_error, mean_absolute_error, r2_score

pd.set_option("display.max_rows", 40)
pd.set_option("display.max_columns", 20)

stud = pd.DataFrame({
    "student_id": range(1, 17),
    "projekt_pkt": [35, 42, 48, 52, 55, 58, 61, 63, 66, 70, 74, 78, 82, 86, 90, 94],
    "quiz_pkt": [40, 45, 50, 55, 44, 60, 58, 62, 65, 68, 72, 75, 80, 84, 88, 92],
    "zadania_pkt": [50, 55, 49, 60, 57, 62, 65, 67, 71, 73, 75, 78, 82, 86, 88, 92],
    "obecnosc_pct": [58, 62, 65, 70, 72, 76, 73, 77, 80, 82, 85, 88, 90, 92, 95, 98],
    "zaliczenie": [
        "nie", "nie", "nie", "tak", "nie", "nie", "tak", "tak",
        "nie", "tak", "tak", "tak", "tak", "tak", "tak", "nie"
    ],
})

features_cls = ["projekt_pkt", "quiz_pkt", "zadania_pkt", "obecnosc_pct"]
target_cls = "zaliczenie"

stud


In [ ]:
print("Rozkład klas:")
display(stud[target_cls].value_counts())

plt.figure(figsize=(7, 4))
colors = stud[target_cls].map({"nie": "#d62728", "tak": "#2ca02c"})
plt.scatter(stud["projekt_pkt"], stud["quiz_pkt"], c=colors, s=80, edgecolor="black")
plt.axvline(59.5, color="black", linestyle="--", label="próg 59.5")
plt.xlabel("projekt_pkt")
plt.ylabel("quiz_pkt")
plt.title("Przykład progu liczbowego: projekt_pkt <= 59.5")
plt.legend()
plt.show()


## 2. Ręczne Gini dla jednego progu

Weź próg:

```text
projekt_pkt <= 59.5
```

Lewa gałąź to studenci z `projekt_pkt <= 59.5`, prawa gałąź to pozostali.

Wzory są te same jak dla cech binarnych:

$$
Gini = 1 - p_{nie}^2 - p_{tak}^2
$$

$$
Gini_{after} = \frac{N_L}{N}Gini_L + \frac{N_R}{N}Gini_R
$$


In [ ]:
threshold = 59.5
left = stud[stud["projekt_pkt"] <= threshold]
right = stud[stud["projekt_pkt"] > threshold]

print("LEFT: projekt_pkt <=", threshold)
display(left[["student_id", "projekt_pkt", "zaliczenie"]])
display(left["zaliczenie"].value_counts())

print("RIGHT: projekt_pkt >", threshold)
display(right[["student_id", "projekt_pkt", "zaliczenie"]])
display(right["zaliczenie"].value_counts())


### Ćwiczenie A

Uzupełnij ręcznie.

To jest nadal **klasyfikacja**, więc liczymy Gini. Nowością jest tylko to, że pytanie powstało przez próg liczbowy:

```text
projekt_pkt <= 59.5 ?
```

**Jak wypełnić krok po kroku:**

```text
1. Lewa gałąź = obserwacje z projekt_pkt <= 59.5.
2. Prawa gałąź = obserwacje z projekt_pkt > 59.5.
3. W każdej gałęzi policz klasy tak/nie.
4. Dla każdej gałęzi policz Gini.
5. Policz ważone Gini_after.
```

Lewa gałąź:

- `nie`: ...
- `tak`: ...
- razem: ...
- $Gini_L = 1 - (liczba\_nie/razem)^2 - (liczba\_tak/razem)^2 = ...$

Prawa gałąź:

- `nie`: ...
- `tak`: ...
- razem: ...
- $Gini_R = 1 - (liczba\_nie/razem)^2 - (liczba\_tak/razem)^2 = ...$

$$
Gini_{after} = \frac{liczba\_lewa}{16}\cdot Gini_L + \frac{liczba\_prawa}{16}\cdot Gini_R = ...
$$


In [ ]:
def gini(labels):
    probs = labels.value_counts(normalize=True)
    return 1 - np.sum(probs ** 2)

parent_gini = gini(stud[target_cls])
gini_left = gini(left[target_cls])
gini_right = gini(right[target_cls])
gini_after = len(left) / len(stud) * gini_left + len(right) / len(stud) * gini_right

print("Gini parent:", round(parent_gini, 6))
print("Gini LEFT:", round(gini_left, 6))
print("Gini RIGHT:", round(gini_right, 6))
print("Gini after:", round(gini_after, 6))
print("Gain:", round(parent_gini - gini_after, 6))


## 3. Jak drzewo znajduje możliwe progi?

Dla cechy liczbowej sortujemy wartości i sprawdzamy progi między kolejnymi wartościami.

Przykład dla `projekt_pkt`:

```text
35, 42, 48, ...
```

Kandydatami są środki między sąsiednimi wartościami:

```text
38.5, 45.0, 50.0, ...
```

Drzewo wybiera próg z najmniejszym `Gini_after`.


In [ ]:
def threshold_table_classification(data, feature, target="zaliczenie"):
    values = np.sort(data[feature].unique())
    thresholds = (values[:-1] + values[1:]) / 2
    rows = []
    parent = gini(data[target])

    for threshold in thresholds:
        left = data[data[feature] <= threshold]
        right = data[data[feature] > threshold]
        weighted = len(left) / len(data) * gini(left[target]) + len(right) / len(data) * gini(right[target])
        rows.append({
            "cecha": feature,
            "prog": threshold,
            "left_n": len(left),
            "right_n": len(right),
            "left_counts": dict(left[target].value_counts()),
            "right_counts": dict(right[target].value_counts()),
            "gini_after": weighted,
            "gain": parent - weighted,
        })

    return pd.DataFrame(rows).sort_values("gini_after").reset_index(drop=True)

threshold_table_classification(stud, "projekt_pkt").round(6)


In [ ]:
all_best = pd.concat([
    threshold_table_classification(stud, feature).head(1)
    for feature in features_cls
], ignore_index=True)

all_best.sort_values("gini_after").round(6)



### Uwaga o remisie progów

Czasem dwa różne progi dają ten sam wynik `Gini_after`. Wtedy implementacja drzewa może wybrać jeden z równorzędnych splitów. To nie jest błąd: ważne jest, że oba splity mają taką samą jakość według kryterium Gini.


In [ ]:

best_gini = all_best["gini_after"].min()
all_best[all_best["gini_after"].eq(best_gini)].sort_values(["cecha", "prog"]).round(6)


## 4. Drzewo klasyfikacyjne na cechach liczbowych

Teraz sprawdzamy, czy `DecisionTreeClassifier` wybiera progi zgodne z obliczeniami.


In [ ]:
X_cls = stud[features_cls]
y_cls = stud[target_cls]

tree_cls = DecisionTreeClassifier(criterion="gini", max_depth=2, random_state=42)
tree_cls.fit(X_cls, y_cls)

print(export_text(tree_cls, feature_names=features_cls))
print("Accuracy na całym małym zbiorze:", accuracy_score(y_cls, tree_cls.predict(X_cls)))

plt.figure(figsize=(11, 5))
plot_tree(
    tree_cls,
    feature_names=features_cls,
    class_names=list(tree_cls.classes_),
    filled=True,
    rounded=True,
    impurity=True,
)
plt.title("Drzewo klasyfikacyjne: progi liczbowe")
plt.show()


## Ważne: progowanie to nie to samo co regresja

Przejście z cech binarnych na cechy liczbowe **nie oznacza jeszcze przejścia do regresji**.

W klasyfikacji również możemy używać progów liczbowych:

```text
projekt_pkt <= 59.5?
```

ale nadal przewidujemy klasę:

```text
zaliczyl = tak / nie
```

Regresja zaczyna się dopiero wtedy, gdy zmienna docelowa jest liczbą, np.:

```text
wynik_pct = 74.2
```

Różnica jest więc taka:

```text
klasyfikacja → przewidujemy klasę, jakość splitu mierzymy np. Gini
regresja     → przewidujemy liczbę, jakość splitu mierzymy np. MSE albo MAE
```

W obu przypadkach węzły drzewa nadal są pytaniami binarnymi typu:

```text
cecha <= próg?  TAK / NIE
```

Najkrócej:

| etap | cechy wejściowe | pytanie w węźle | kryterium splitu | co przewiduje liść |
|---|---|---|---|---|
| Notebook 4.1 | binarne | `cecha = 1?` albo `cecha <= 0.5?` | Gini | klasa |
| Notebook 4.2A | liczbowe/punktowe | `cecha <= próg?` | Gini | klasa |
| Notebook 4.2B | liczbowe/punktowe | `cecha <= próg?` | MSE / MAE | liczba |


## 5. Drzewo regresyjne: z klas do liczb

W klasyfikacji liść zwraca klasę, np. `tak` albo `nie`.

W regresji liść zwraca liczbę. Najczęściej jest to średnia wartość `y` w liściu.

To jest główna zmiana w części B:

```text
klasyfikacja:  próg → grupy → Gini → klasa
regresja:      próg → grupy → MSE  → liczba
```

Sam próg nie jest nowością regresji. Próg był już w klasyfikacji na cechach liczbowych. Nowością regresji jest to, że:

1. przewidujemy liczbę,
2. jakość splitu mierzymy błędem wartości liczbowych, np. MSE,
3. przewidywanie w liściu jest zwykle średnią wartości `y` w tym liściu.

Jakość splitu mierzymy nie przez Gini, tylko przez rozrzut wartości liczbowych, np. MSE:

$$
MSE = \frac{1}{n}\sum_i (y_i - \bar{y})^2
$$


In [ ]:
reg_df = pd.DataFrame({
    "student_id": range(1, 13),
    "godziny_nauki": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "powtorki": [0, 0, 1, 1, 2, 2, 3, 3, 4, 5, 5, 6],
    "wynik_pct": [35, 38, 42, 50, 55, 57, 64, 70, 73, 78, 84, 88],
})

reg_df


In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(reg_df["godziny_nauki"], reg_df["wynik_pct"], s=80, edgecolor="black")
plt.axvline(6.5, color="black", linestyle="--", label="próg 6.5")
plt.xlabel("godziny_nauki")
plt.ylabel("wynik_pct")
plt.title("Drzewo regresyjne: próg dla cechy liczbowej")
plt.legend()
plt.show()


### Ćwiczenie B

Dla progu `godziny_nauki <= 6.5` policz ręcznie jakość splitu regresyjnego.

Tutaj mamy już **regresję**, więc nie liczymy Gini. Liczymy błąd predykcji liczbowej, np. MSE.

Lewy liść zawiera wyniki:

```text
35, 38, 42, 50, 55, 57
```

Prawy liść zawiera wyniki:

```text
64, 70, 73, 78, 84, 88
```

**Jak wypełnić tabelę:**

```text
DLA każdego liścia:
    1. policz liczbę obserwacji n
    2. policz sumę wyników
    3. policz średnią = suma wyników / n
    4. dla każdego wyniku policz odchylenie = wynik - średnia
    5. podnieś każde odchylenie do kwadratu
    6. zsumuj kwadraty odchyleń: to jest SSE
    7. policz MSE = SSE / n
```

Uzupełnij tabelę:

| liść | liczba obserwacji | suma wyników | średnia liścia | suma kwadratów odchyleń, SSE | MSE |
|---|---:|---:|---:|---:|---:|
| lewy | ... | ... | ... | ... | ... |
| prawy | ... | ... | ... | ... | ... |

Następnie policz ważone MSE po splicie:

$$
MSE_{after} = \frac{liczba\_lewa}{12}\cdot MSE_L + \frac{liczba\_prawa}{12}\cdot MSE_R = ...
$$

Wskazówka: w każdym liściu predykcją drzewa regresyjnego jest średnia z tego liścia. MSE liczymy jako średni kwadrat błędu względem tej średniej.



### Minićwiczenie kodowe po B. MSE tak jak na kartce

Teraz zakoduj dokładnie te same kroki: średnią liścia, MSE lewego liścia, MSE prawego liścia i ważone MSE po splicie.

**Pseudokod:**

```text
mse_student(values):
    mean = średnia(values)
    deviations = values - mean
    squared_errors = deviations^2
    return średnia(squared_errors)

regression_split_score_student(data, feature, threshold):
    left_part = obserwacje, gdzie feature <= threshold
    right_part = obserwacje, gdzie feature > threshold

    mse_left = mse_student(wyniki w left_part)
    mse_right = mse_student(wyniki w right_part)

    weight_left = liczba_obserwacji_left / liczba_obserwacji_całość
    weight_right = liczba_obserwacji_right / liczba_obserwacji_całość

    weighted_mse = weight_left * mse_left + weight_right * mse_right
    zwróć wyniki w słowniku
```


In [ ]:

# Ćwiczenie B-kod: zaimplementuj MSE tak, jak zostało policzone ręcznie.
#
# Cel: kod ma odtworzyć tabelę z ćwiczenia ręcznego:
# - średnia lewego liścia,
# - MSE lewego liścia,
# - średnia prawego liścia,
# - MSE prawego liścia,
# - ważone MSE po splicie.


def mse_student(values):
    # values to jedna kolumna z wartościami liczbowymi, np. wynik_pct w jednym liściu.

    # KROK 1. Policz średnią w liściu.
    # To jest predykcja drzewa regresyjnego dla tego liścia.
    mean = values.mean()

    # TODO 1: policz odchylenia każdej wartości od średniej.
    # Wzór:
    #     deviations = values - mean
    deviations = ...

    # TODO 2: podnieś odchylenia do kwadratu.
    # Wzór:
    #     squared_errors = deviations ** 2
    squared_errors = ...

    # TODO 3: zwróć średnią z kwadratów błędów.
    # Wzór:
    #     MSE = średnia(squared_errors)
    # Podpowiedź: użyj squared_errors.mean() albo np.mean(squared_errors).
    return ...


def regression_split_score_student(data, feature, threshold, target="wynik_pct"):
    # KROK 1. Podziel dane progiem liczbowym na lewą i prawą gałąź.
    left_part = data[data[feature] <= threshold]
    right_part = data[data[feature] > threshold]

    # TODO 4: policz MSE lewego liścia.
    # Użyj funkcji mse_student na kolumnie target w left_part.
    # Podpowiedź:
    #     mse_student(left_part[target])
    mse_left = ...

    # TODO 5: policz MSE prawego liścia.
    # Podpowiedź:
    #     mse_student(right_part[target])
    mse_right = ...

    # TODO 6: policz wagę lewego liścia.
    # Wzór:
    #     weight_left = len(left_part) / len(data)
    weight_left = ...

    # TODO 7: policz wagę prawego liścia.
    # Wzór:
    #     weight_right = len(right_part) / len(data)
    weight_right = ...

    # TODO 8: policz ważone MSE po splicie.
    # Wzór:
    #     weighted_mse = weight_left * mse_left + weight_right * mse_right
    weighted_mse = ...

    return {
        "prog": threshold,
        "left_n": len(left_part),
        "right_n": len(right_part),
        "mean_left": left_part[target].mean(),
        "mean_right": right_part[target].mean(),
        "mse_left": mse_left,
        "mse_right": mse_right,
        "mse_after": weighted_mse,
    }


if mse_student(pd.Series([1, 3])) is Ellipsis:
    print("TODO: uzupełnij mse_student i regression_split_score_student.")
else:
    result = regression_split_score_student(reg_df, "godziny_nauki", 6.5)
    if any(value is Ellipsis for value in result.values()):
        print("TODO: uzupełnij wszystkie wartości w regression_split_score_student.")
    else:
        display(pd.Series(result).round(4))


In [ ]:
# Komórka kontrolna po minićwiczeniu B-kod.
#
# Funkcja mse poniżej będzie też używana w dalszej części notebooka do szukania progów.
# Tabela kontrolna pokazuje się dopiero wtedy, gdy uzupełnisz własne funkcje
# mse_student i regression_split_score_student.


def mse(values):
    mean = values.mean()
    return np.mean((values - mean) ** 2)


threshold_reg = 6.5
left_reg = reg_df[reg_df["godziny_nauki"] <= threshold_reg]
right_reg = reg_df[reg_df["godziny_nauki"] > threshold_reg]

mse_left = mse(left_reg["wynik_pct"])
mse_right = mse(right_reg["wynik_pct"])
weighted_mse = len(left_reg) / len(reg_df) * mse_left + len(right_reg) / len(reg_df) * mse_right


def b_reg_kod_uzupelniony():
    """Sprawdza, czy minićwiczenie B-kod zostało uzupełnione."""
    try:
        if mse_student(pd.Series([1, 3])) is Ellipsis:
            return False
        result = regression_split_score_student(reg_df, "godziny_nauki", threshold_reg)
        return not any(value is Ellipsis for value in result.values())
    except Exception:
        return False


if not b_reg_kod_uzupelniony():
    print("Najpierw uzupełnij minićwiczenie B-kod. Potem ta komórka pokaże kontrolę MSE.")
else:
    result_student = regression_split_score_student(reg_df, "godziny_nauki", threshold_reg)
    print("Średnia LEFT:", round(left_reg["wynik_pct"].mean(), 2))
    print("Średnia RIGHT:", round(right_reg["wynik_pct"].mean(), 2))
    print("MSE LEFT:", round(mse_left, 2))
    print("MSE RIGHT:", round(mse_right, 2))
    print("MSE after:", round(weighted_mse, 2))
    print("Czy MSE_after studenta zgodne?:", np.isclose(result_student["mse_after"], weighted_mse))

In [ ]:
def threshold_table_regression(data, feature, target="wynik_pct"):
    values = np.sort(data[feature].unique())
    thresholds = (values[:-1] + values[1:]) / 2
    rows = []
    parent = mse(data[target])

    for threshold in thresholds:
        left = data[data[feature] <= threshold]
        right = data[data[feature] > threshold]
        weighted = len(left) / len(data) * mse(left[target]) + len(right) / len(data) * mse(right[target])
        rows.append({
            "cecha": feature,
            "prog": threshold,
            "left_n": len(left),
            "right_n": len(right),
            "mean_left": left[target].mean(),
            "mean_right": right[target].mean(),
            "mse_after": weighted,
            "mse_gain": parent - weighted,
        })

    return pd.DataFrame(rows).sort_values("mse_after").reset_index(drop=True)

threshold_table_regression(reg_df, "godziny_nauki").round(3)


## 6. Schodki drzewa regresyjnego

Drzewo regresyjne daje funkcję schodkową: w każdym liściu przewiduje jedną stałą wartość.

Zwiększanie `max_depth` daje więcej schodków, ale może prowadzić do przeuczenia.


In [ ]:
X_reg = reg_df[["godziny_nauki"]]
y_reg = reg_df["wynik_pct"]
x_grid = pd.DataFrame({"godziny_nauki": np.linspace(1, 12, 300)})

for depth in [1, 2, 4]:
    tree_reg = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree_reg.fit(X_reg, y_reg)
    pred_grid = tree_reg.predict(x_grid)

    plt.figure(figsize=(7, 4))
    plt.scatter(reg_df["godziny_nauki"], y_reg, s=80, edgecolor="black", label="dane")
    plt.plot(x_grid["godziny_nauki"], pred_grid, linewidth=2, label="predykcja drzewa")
    plt.xlabel("godziny_nauki")
    plt.ylabel("wynik_pct")
    plt.title(f"Drzewo regresyjne, max_depth={depth}")
    plt.legend()
    plt.show()


## Pytania końcowe i schemat całej ścieżki

Odpowiedz własnymi słowami:

1. Czy próg liczbowy zmienia ideę drzewa, czy tylko typ pytania?
2. Dlaczego w Notebooku 4.1 nie trzeba było szukać progów?
3. Co minimalizujemy w drzewie klasyfikacyjnym?
4. Co minimalizujemy w drzewie regresyjnym?
5. Co zwraca liść w klasyfikacji?
6. Co zwraca liść w regresji?
7. Dlaczego `projekt_pkt <= 59.5` może wystąpić zarówno w klasyfikacji, jak i w regresji?

Schemat:

```text
Notebook 4.1
────────────
cechy binarne
      ↓
gotowe pytania TAK/NIE
      ↓
Gini
      ↓
klasa

Notebook 4.2A
─────────────
cechy liczbowe / punktowe
      ↓
progowanie: cecha <= próg?
      ↓
Gini
      ↓
klasa

Notebook 4.2B
─────────────
cechy liczbowe / punktowe
      ↓
progowanie: cecha <= próg?
      ↓
MSE / MAE
      ↓
liczba
```

Najważniejsze zdanie:

> **Progowanie rozwiązuje problem cech liczbowych, a regresja oznacza przewidywanie liczby. To są dwie różne sprawy.**


## Łącznik do Notebooka 4.3: dlaczego jedno drzewo nie zawsze wystarcza

Po Notebooku 4.1 i 4.2 umiemy już zbudować pojedyncze drzewo:

```text
cecha -> pytanie/prog -> Gini albo MSE -> najlepszy split -> liście
```

Pojedyncze drzewo jest bardzo interpretowalne, ale ma jedną ważną wadę: może być **niestabilne**. Mała zmiana danych treningowych może zmienić pierwszy lub drugi split, a przez to całe drzewo.

Następny notebook nie zmienia mechaniki pojedynczego splitu. Zmienia sposób korzystania z drzew:

```text
zamiast jednego drzewa
      ↓
budujemy wiele trochę różnych drzew
      ↓
agregujemy ich decyzje
      ↓
dostajemy stabilniejszy model
```

To prowadzi do Random Forest.

Najważniejsze rozróżnienie przed kolejnym notebookiem:

| Element | Pojedyncze drzewo | Random Forest |
|---|---|---|
| Split w węźle | Gini/MSE i próg tak jak wcześniej | Gini/MSE i próg tak jak wcześniej |
| Dane dla modelu | zwykle jeden zbiór treningowy | wiele próbek bootstrapowych |
| Liczba drzew | jedno | wiele |
| Wynik | jeden liść / jedna decyzja | głosowanie lub średnia z wielu drzew |
| Główna korzyść | interpretowalność | większa stabilność, mniejsza wariancja |